# core

> Basic server-startup stuff

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, subprocess, sys, time, urllib.request, urllib.error
from pathlib import Path
from typing import Annotated
from fastcore.script import call_parse

def _port_owner(port):
    "Return 'boopiter' if boopiter answers on `port`, 'other' if something else does, None if nothing does."
    try:
        with urllib.request.urlopen(f'http://localhost:{port}/_boopiter_ping', timeout=1.0) as r:
            text = r.read().decode().strip()
        return 'boopiter' if text == 'boopiter' else 'other'
    except urllib.error.HTTPError:
        return 'other'   # got a real HTTP response, just not our ping -- something else is there
    except (ConnectionRefusedError, urllib.error.URLError):
        return None      # nothing is accepting connections there
    except Exception:
        return 'other'   # some other failure mode -- be conservative, don't touch it

def _kill_port(port):
    try:
        pids = subprocess.run(['lsof', '-ti', f'tcp:{port}'], capture_output=True, text=True).stdout.split()
    except FileNotFoundError:
        pids = []
    for pid in pids: subprocess.run(['kill', '-9', pid])
    if pids: time.sleep(0.3)

def _build_tailwind():
    "Precompile a static Tailwind CSS file once per launch, replacing the CDN's in-browser JIT compiler (which recompiles on every htmx DOM update -- a major source of per-interaction lag)."
    pkg_dir = Path(__file__).parent
    inp, out = pkg_dir/'static/tw_input.css', pkg_dir/'static/tailwind.css'
    venv_bin = Path(sys.executable).parent/'tailwindcss'  # pytailwindcss installs its binary alongside python, not necessarily on PATH
    exe = str(venv_bin) if venv_bin.exists() else 'tailwindcss'
    try:
        subprocess.run([exe, '-i', str(inp), '-o', str(out), '--minify', '--cwd', str(pkg_dir)],
                       capture_output=True, timeout=60, check=True)
    except Exception as e:
        print(f"Tailwind precompile failed ({e}); falling back to the slower CDN JIT build.", file=sys.stderr)

@call_parse
def launch(
    nbfile: Annotated[str, {'opt': False, 'nargs': '?'}] = None,  # .ipynb file to load on startup
    port: int = 8000,  # the port to serve boopiter on
):
    "Launch (or relaunch) the boopiter server; refuses to touch a port held by something that isn't boopiter"
    owner = _port_owner(port)
    if owner == 'other':
        print(f"Port {port} is already in use by something that isn't boopiter -- aborting.", file=sys.stderr)
        sys.exit(1)
    if owner == 'boopiter':
        print(f"Killing previous boopiter instance on port {port}...")
        _kill_port(port)
    _build_tailwind()
    if nbfile:
        from . import cells as _cells
        try:
            _cells.load_notebook(nbfile)
        except FileNotFoundError:
            print(f"No such notebook: {nbfile} -- starting with a blank notebook instead.", file=sys.stderr)
    os.environ['BOOPITER_PORT'] = str(port)  # lets a self-restart (see cells.restart_server) rebind the same port
    import uvicorn
    uvicorn.run('boopiter.cells:app', host='0.0.0.0', port=port)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()